# Capital por activo: oferta ≠ adquisición ≠ gasto mensual

## tl;dr
Dos fichas de tracto usado MX/US conservan precios anunciados, no costos capitalizables. El contraste SFI permite comprobar 52 semanas / 12 meses, pero no cotizar capital completo.

## Context & Methods
Se reutilizan ADR-008/009 y los expedientes B1/B2. Sólo documentación pública, sin FX ni valores adoptados. Captura 2026-09-29 no certifica vigencia comercial ni exposición del piloto.

### Key Assumptions
El perfil propio/usado es propuesta. Moneda US no explícita: los cálculos de pagos mantienen unidades monetarias del signo publicado, no USD certificados. No aplicamos vida fiscal, vida de nuevo, garantía ni porcentaje de residual de costo original a una compra usada.

## Data
B2_CAPITAL_ACQUISITION_BRIDGE_2026-09-29.json contiene enlaces, stock, importes originales, límites y hashes disponibles. No hay distribución de mercado. Schneider fue leído vía texto web; descarga HTML falló, por lo que no se inventa hash.

## Results
Aritmética y salvaguardas reproducibles abajo. Ejecutar desde la raíz del worktree.


In [1]:
import json
from decimal import Decimal
from pathlib import Path
import hashlib

# Run from repository root. No network, credentials, database or engine writes.
research = Path("docs/research")
d = json.loads((research / "B2_CAPITAL_ACQUISITION_BRIDGE_2026-09-29.json").read_text(encoding="utf-8"))
checks = []
def check(name, condition):
    assert condition, name
    checks.append(name)
    print("PASS:", name)

mx, us = d["acquisitionCards"]
lease = d["leaseContrast"]
check("MX displayed rounding is 0.14 MXN, not a price change",
      Decimal(str(mx["askingPrice"]["headerRoundedValue"])) - Decimal(str(mx["askingPrice"]["value"])) == Decimal("0.14"))
check("52 published weeks yield 35,880 monetary units of base payments",
      lease["weeklyBasePayment"] * lease["leaseWeeks"] == lease["annualBasePayment"] == 35880)
check("Annual-average month is 2,990, not a four-week month",
      lease["annualBasePayment"] / 12 == lease["averageMonthlyBasePayment"] == 2990)
check("Four weeks yield 2,760; monthly-average difference is 230",
      lease["weeklyBasePayment"] * 4 == lease["fourWeekPayment"] == 2760
      and lease["averageMonthlyBasePayment"] - lease["fourWeekPayment"] == 230)
check("US dollar context remains inference, not explicit currency evidence",
      us["askingPrice"]["currency"] is None and lease["currency"] is None)
check("Two owned-used reference cards retain unknown cost/life/residual/interest",
      all(c["condition"] == "USED" and all(c[k] is None for k in [
          "capitalizableAcquisitionCost", "remainingHorizonMonths", "netResidualAtEnd",
          "monthlyDepreciation", "monthlyInterestExpense"]) for c in d["acquisitionCards"]))
check("Unavailable TRAC contrast is not an all-in lease quote or owned cost",
      lease["availableUnits"] == 0 and not lease["usableAsCurrentLeaseQuote"]
      and not lease["usableInOwnedProfile"] and lease["allInMonthlyCost"] is None)
check("Same shared trailer is unknown, not zero and not charged twice",
      d["sharedTrailer"]["monthlyCapitalCost"] is None
      and not d["sharedTrailer"]["costKnownZero"] and not d["sharedTrailer"]["chargedFullToBothLegs"])
check("No calibrated economic output, adoption or closure",
      not d["calibrated"] and d["adoptedValues"] == 0
      and all(v is None for v in d["outputs"].values()) and d["decision"]["businessGatesClosed"] == 0)
for item in d["unchangedFiles"]:
    check("Unchanged " + item["file"],
          hashlib.sha256((research / item["file"]).read_bytes()).hexdigest() == item["sha256"])
print("Checks:", len(checks))
print("CFU/CVU and governed capital remain unknown; payments above are base-payment arithmetic only.")


PASS: MX displayed rounding is 0.14 MXN, not a price change
PASS: 52 published weeks yield 35,880 monetary units of base payments
PASS: Annual-average month is 2,990, not a four-week month
PASS: Four weeks yield 2,760; monthly-average difference is 230
PASS: US dollar context remains inference, not explicit currency evidence
PASS: Two owned-used reference cards retain unknown cost/life/residual/interest
PASS: Unavailable TRAC contrast is not an all-in lease quote or owned cost
PASS: Same shared trailer is unknown, not zero and not charged twice
PASS: No calibrated economic output, adoption or closure
PASS: Unchanged B2_PAYROLL_CAPITAL_CONTRACT_TEMPLATE_2026-09-29.json
PASS: Unchanged B2_PUBLIC_REFERENCE_COMPONENTS_LEDGER_2026-09-29.json
PASS: Unchanged B2_REFERENCE_PROFILE_EXPOSURE_2026-09-29.json
Checks: 12
CFU/CVU and governed capital remain unknown; payments above are base-payment arithmetic only.


## Takeaways

690 × 52 / 12 = 2,990 unidades monetarias de pago base promedio mensual; 690 × 4 = 2,760 no representa ese promedio anual. Ni depósito, reserva de mantenimiento, interés, seguro ni saldo final están incluidos o asumidos cero.

El puente al motor sigue simbólico: depreciación mensual = (adquisición capitalizable − residual neto) / meses restantes; más interés, sin principal duplicado. CFU requiere además presupuesto y exposición del mismo alcance/periodo. No se crea fórmula activa ni motor paralelo.

B2 parcial, B1/E1 abiertos y 0/4 gates. Próxima decisión acotada: escenarios públicos de vida restante/residual por usado y quién soporta el remolque compartido. Validación de ejecución no equivale a calibración económica ni aceptación autenticada.

Execution note: standard-library Python execution with stored outputs, not a Jupyter kernel. Rendered notebook inspection remains pending.
